Aprobar modelo `challenger` a `champion`

**Job:** Usualmente este notebook es ejecutado como parte de un job automatico. Pasos del job
1. Evaluacion: 
2. Aprobacion: base_parameters={"approval_tag_name": "Approval_Check"}
3. Despliegue: base_parameters={"smoke_test": False}

Parametros del job:
- name="model_full_name", default=main.dbdemos_fs_travel.purchase_travel_classifier
- name="model_version", default=1

## 1. Configuración

In [0]:
import sys
python_version = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
print(f"Versión de Python actual: {python_version}")

Versión de Python actual: 3.12.3


In [0]:
catalog = "medallion_dev"
schema = "gold"
model_name = "customer_churn_classifier"

dbutils.widgets.text("model_full_name", f"{catalog}.{schema}.{model_name}", "Modelo nombre completo: catalog.schema.name") 
dbutils.widgets.text("model_version", "1", "Modelo Version (@challenger)")
dbutils.widgets.text("approval_tag_name", "Approval_Check", "Tag chequeo aprobacion")

In [0]:
model_full_name = dbutils.widgets.get("model_full_name")
model_version = int(dbutils.widgets.get("model_version"))
tag_name = dbutils.widgets.get("approval_tag_name") 
catalog =  model_full_name.split('.')[0]
schema =  model_full_name.split('.')[1]
model_name =  model_full_name.split('.')[-1]
model_alias_challenger = "challenger" 
model_alias_champion = "champion"
delete_other_aliases = True

In [0]:
import mlflow
from mlflow.tracking.client import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")

model_details = client.get_model_version(model_full_name, model_version)
if not model_details:
    raise Exception("No encontrado version del modelo registrado en UC.")

tags = model_details.tags
aliases = model_details.aliases

# Check if any tag matches the approval tag name
if not any(tag == tag_name for tag in tags.keys()):
  raise Exception("Version del modelo no aprobada para despliegue")
else:
  if tags.get(tag_name).lower() == "approved":
    print("Version del modelo aprobada para despliegue")
    
    if model_alias_champion.lower() in [alias.lower() for alias in aliases]:
      print(f"El modelo ya tiene el alias: {model_alias_champion}. Ya fue promovido a esta etapa.")
    else:
      print(f"El modelo No tiene el alias: {model_alias_champion}. Será promovido a esta etapa!")
      
      if delete_other_aliases:
        try:
          for alias in aliases:
              if alias.lower() != model_alias_champion.lower():
                  client.delete_registered_model_alias(name=model_full_name, alias=alias)
        except:
          pass

      client.set_registered_model_alias(name=model_full_name, alias=model_alias_champion, version=model_version)
  else:
    raise Exception("Version del modelo no aprobada para despliegue")

Version del modelo aprobada para despliegue
El modelo No tiene el alias: Champion. Será promovido a esta etapa!
